In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import graphinglib as gl
import pyregion
from tqdm import tqdm

from src.tools.plotting import *
from src.hdu.grouped_maps import GroupedMaps
from src.hdu.map import Map
from src.hdu.cube import Cube
from src.hdu.header import Header
from src.tools.loki_models import LOKIModels
from src.tools.statistics.advanced_stats import structure_function, get_fitted_structure_function_figure

# Kill the FITSFixedWarnings
from warnings import simplefilter
from astropy.wcs import FITSFixedWarning
simplefilter("ignore", category=FITSFixedWarning)

In [ ]:
# Data loading

large_fwhm_gm = GroupedMaps.load(
    "data/loki/output_NGC4696_G235H_F170LP_QOBr_tied_global_m1_lores_fewlines_largeFWHM/"
    "NGC4696_G235H_F170LP_QOBr_tied_global_m1_lores_fewlines_largeFWHM_parameter_maps.fits",
    mute=True,
)
data_continuum_gm = GroupedMaps.load(
    "data/loki/data_continuum_parameter_maps_updated_wcs.fits",
    mute=True,
)
lores_header = Cube.load("data/wicked/ngc4696_v7_s3d_choiclip1_1_marquis1_updatedwcs_lores.fits", hdu_index=1).header

hires_data_continuum_gm = GroupedMaps.load(
    "data/loki/hires_data_continuum_parameter_maps_updated_wcs.fits",
    mute=True,
)

large_fwhm_models = LOKIModels.load_from_version("large_fwhm")
data_continuum_models = LOKIModels.load_from_version("data_continuum")

# MUSE data
# muse_gm = GroupedMaps.load("data/muse/NGC4696_MUSE_MAPS.fits")
# for map_ in muse_gm.maps:
#     map_.header["RADESYS"] = "FK5"  # add the missing RADESYS keyword so that the WCS transformation works correctly
# MUSE_VSYS = float(muse_gm.header["vsys"])  # in km/s

In [ ]:
# Useful constants

SNR_cut = 2
pc_per_arcsec = 206
PSF_pc = 0.17 * pc_per_arcsec  # PSF in arcsec (0.17) obtained from G. C. Jones et al. (2026)

bold_ref_label_format = lambda text: rf"$\textbf{{{text}}}$"  # for making bold reference labels
ref_label_params = {"highlight_color": "white", "highlight_alpha": 0.7, "format": bold_ref_label_format}

# Tests

## Lores vs hires

In [ ]:
reg = pyregion.open(f"data/vsf/filament_3.reg")
ellipse_mean_radius_arcsec = 1.0945
max_scale = pc_per_arcsec * ellipse_mean_radius_arcsec

In [ ]:
lores_velocity_map = data_continuum_gm["LINES.HI_PA_ALPHA.1.VOFF"]
lores_snr = data_continuum_gm["LINES.HI_PA_ALPHA.1.SNR"].data
hires_velocity_map = hires_data_continuum_gm["LINES.HI_PA_ALPHA.1.VOFF"]
hires_snr = hires_data_continuum_gm["LINES.HI_PA_ALPHA.1.SNR"].data

figs = []
for velocity_map, snr, arcsec_per_px in zip(
    [lores_velocity_map, hires_velocity_map],
    [lores_snr, hires_snr],
    [0.1, 0.05],
):
    velocity_region = velocity_map.get_masked_region(reg).mask(snr < SNR_cut).crop_nans()
    str_func = structure_function(velocity_region.data, 1)
    print(str_func)
    raise
    str_func[:, 0] *= pc_per_arcsec * arcsec_per_px  # convert from pixels to pc
    figs.append(get_fitted_structure_function_figure(str_func, fit_bounds=(PSF_pc, max_scale), number_of_iterations=1000))

pa_alpha_fig = figs[0].copy_with(elements=[
    figs[0][0][0].copy_with(label="Low-resolution data"),
    figs[1][0][0].copy_with(face_color="green", label="High-resolution data"),
    figs[0][0][1].copy_with(color="grey", alpha=0.5),
    figs[1][0][1].copy_with(color="lime", alpha=0.5),
], annotations=[gl.Text(0.99, 0.96, r"Pa$\alpha$ VSF", h_align="right")])
# pa_alpha_fig.save("figures/paper_2/tests/lores_vs_hires_pa_alpha.pdf")

In [ ]:
lores_velocity_map = data_continuum_gm["LINES.H210_S1.1.VOFF"]
lores_snr = data_continuum_gm["LINES.H210_S1.1.SNR"].data
hires_velocity_map = hires_data_continuum_gm["LINES.H210_S1.1.VOFF"]
hires_snr = hires_data_continuum_gm["LINES.H210_S1.1.SNR"].data

figs = []
for velocity_map, snr, arcsec_per_px in zip(
    [lores_velocity_map, hires_velocity_map],
    [lores_snr, hires_snr],
    [0.1, 0.05],
):
    velocity_region = velocity_map.get_masked_region(reg).mask(snr < 5).crop_nans()
    if arcsec_per_px == 0.05: gl.SmartFigure(elements=[velocity_region.data.plot]).show()#; raise
    str_func = structure_function(velocity_region.data, 1, log_bin_width=0.015, bin_start=5)
    str_func[:, 0] *= pc_per_arcsec * arcsec_per_px  # convert from pixels to pc
    figs.append(get_fitted_structure_function_figure(str_func, fit_bounds=(PSF_pc, max_scale), number_of_iterations=1000))

s1_fig = figs[0].copy_with(elements=[
    figs[0][0][0].copy_with(label="Low-resolution data"),
    figs[1][0][0].copy_with(face_color="green", label="High-resolution data"),
    figs[0][0][1].copy_with(color="grey", alpha=0.5),
    figs[1][0][1].copy_with(color="lime", alpha=0.5),
], annotations=[gl.Text(0.99, 0.96, r"S(1) VSF", h_align="right")])
# s1_fig.save("figures/paper_2/tests/lores_vs_hires_s1.pdf")
# s1_fig.save("figures/paper_2/tests/lores_vs_hires_s1_log.pdf")

## Regions

In [ ]:
velocity_map = hires_data_continuum_gm["LINES.HI_PA_ALPHA.1.VOFF"]
snr = hires_data_continuum_gm["LINES.HI_PA_ALPHA.1.SNR"].data

figs = []
regions = ["core_a", "core_b", "swirl", "filament_a", "filament_b", "filament_c", "background", "whole"]#[3:4]
for region in tqdm(regions):
    reg = pyregion.open(f"data/regions/{region}.reg")
    velocity_region = velocity_map.get_masked_region(reg).mask(snr < SNR_cut).crop_nans()
    # gl.SmartFigure(elements=[velocity_region.data.plot]).show(); raise
    str_func = structure_function(velocity_region.data, 1)
    str_func[:, 0] *= pc_per_arcsec * 0.05  # convert from pixels to pc
    figs.append(
        get_fitted_structure_function_figure(str_func, fit_bounds=(PSF_pc, 300), number_of_iterations=1000)
        .copy_with(title=f"{region}, n_pix$={velocity_region.get_statistics(mute=True)["nbpixels"]}$",
                   reference_labels_loc="inside")
        .set_reference_labels_params(**ref_label_params)
    )

fig = gl.SmartFigure(4, 2, size=(9, 11), elements=figs)
# fig.save("figures/paper_2/tests/pa_alpha.pdf")

In [ ]:
import numpy as np
from src.tools.statistics.advanced_stats import structure_function

arr = np.array([
    [1., 2., 3.],
    [4., 5., 6.],
    [7., 8., 9.],
])

structure_function(arr, 1, 1.)

## Stars